In [1]:
import numpy as np
import pandas as pd
import rasterio as rio
from rasterio.warp import reproject, Resampling
from scipy.optimize import curve_fit
from scipy.stats import t
from matplotlib import pyplot as plt
from tqdm.autonotebook import tqdm

C:\Users\Administrator\AppData\Local\Temp\ipykernel_32004\3581257061.py:8: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


In [2]:
names = locals()

In [ ]:
country_raster_1km_    = rio.open('../results/country-rasters/country-raster-eck4-1km.tif')
country_raster_50km_   = rio.open('../results/country-rasters/country-raster-eck4-50km.tif')

country_raster_50km    = country_raster_50km_.read(1)
mining_suitability_1km = rio.open('../results/mining-suitability/global-mining-suitability-1km.tif').read(1)
available_area_BAU_1km = rio.open('../results/mining-area/available-area-BAU-1km.tif').read(1)
available_area_CON_1km = rio.open('../results/mining-area/available-area-CON-1km.tif').read(1)

extraction-area relationship

In [4]:
DF_1 = pd.read_csv('../results/extraction-area-relationship.csv')

In [5]:
x_data = DF_1['E (Mt)'].values
y_data = DF_1['A (ha)'].values

In [6]:
def func_1(x, a, b):
    return a * x ** b

In [7]:
popt, pcov = curve_fit(func_1, x_data, y_data)
a, b = popt

In [11]:
all_tau = {'L': 5000, 'M': 6000, 'H': 7000}

In [12]:
def output_tif(path, array, template_raster):
    """Write a 2D array to GeoTIFF following the metadata of a template raster."""
    with rio.open(
        path,
        'w',
        driver='GTiff',
        height=array.shape[0],
        width=array.shape[1],
        count=1,
        dtype=array.dtype,
        crs=template_raster.crs,
        transform=template_raster.transform,
        compress='lzw'
    ) as dst:
        dst.write(array, 1)

In [ ]:
for L_1 in tqdm(['L', 'M', 'H']):
    tau = all_tau[L_1]

    for scenario in tqdm(['BAU', 'CON']):
        available_area_1km = names[f'available_area_{scenario}_1km']
        mask_valid = (available_area_1km > 0) & (mining_suitability_1km >= tau)
        cand_area_1km = np.where(mask_valid, available_area_1km, 0)

        # capacity_1km_mid = func_1(cand_area_1km*100, a_2, b_2)
        # capacity_1km_lower, capacity_1km_upper = predict_interval(cand_area_1km*100, popt_2, pcov_2, dof_2)

        capacity_1km_mid = (cand_area_1km*100 / a) ** (1 / b)
        capacity_1km_lower = (cand_area_1km*100 / a) ** (1 / (b * 1.05))
        capacity_1km_upper = (cand_area_1km*100 / a) ** (1 / (b * 0.95))

        capacity_1km_list = [capacity_1km_lower, capacity_1km_mid, capacity_1km_upper]

        for L_2, capacity_1km in tqdm(zip(['L', 'M', 'H'], capacity_1km_list)):
            capacity_50km = np.zeros(country_raster_50km_.shape, dtype=np.float32)
            reproject(
                source=capacity_1km,
                destination=capacity_50km,
                src_transform=country_raster_1km_.transform,
                src_crs=country_raster_1km_.crs,
                dst_transform=country_raster_50km_.transform,
                dst_crs=country_raster_50km_.crs,
                resampling=Resampling.sum
            )

            path_1km = f'../results/uncertainty-analysis/extraction-capacity-{scenario}-1km-{L_1}{L_2}.tif'
            path_50km = f'../results/uncertainty-analysis/extraction-capacity-{scenario}-50km-{L_1}{L_2}.tif'

            output_tif(path_1km, capacity_1km, country_raster_1km_)
            output_tif(path_50km, capacity_50km, country_raster_50km_)
   